# Converting Lara's GRB FITS files to a gammapy-compatible format

In [8]:
from pathlib import Path
from datetime import datetime

In [9]:
catalog = Path('/Users/jarred/Downloads/TeV Catalog O5')

list_of_files = list(catalog.glob('*.fits'))

print(f"Found {len(list_of_files)} files")



Found 2307 files


## Example Processing from Fabio's Notebook

In [ ]:
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.table import Table
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.coordinates import Distance
from regions import PointSkyRegion
from gammapy.data import FixedPointingInfo
from gammapy.maps import MapAxis, RegionNDMap
from gammapy.modeling.models import (
    LightCurveTemplateTemporalModel,
)

import gammapy
print(f"Gammapy version: {gammapy.__version__}")

# filter future date warnings for this project
import warnings
from erfa import ErfaWarning
warnings.filterwarnings('ignore', category=ErfaWarning)

Gammapy version: 2.0.1


In [22]:
filepath = sorted(list_of_files)[1533]
energies = Table.read(filepath, format="fits", hdu=1)
times = Table.read(filepath, format="fits", hdu=2)
spectra = Table.read(filepath, format="fits", hdu=3)

# Get header information from primary HDU
header = fits.getheader(filepath, ext=0)

latitutde = header['LAT'] * u.rad
longitude = header['LONG'] * u.rad
distance = (header['DISTANCE'] * u.kpc).to(u.Mpc)
ra = longitude.to(u.deg)
dec = latitutde.to(u.deg)

print(f"Number of energies: {len(energies)}. \nNumber of times: {len(times)}")
print(f"RA, Dec: {round(ra, 2)}, {round(dec, 2)}, Distance: {distance}")

Number of energies: 41. 
Number of times: 70
RA, Dec: 273.93 deg, -0.86 deg, Distance: 843.0 Mpc


Get the event timestamp(s)

There are two timestamps for each event, since each spectral model will be used twice for more statistics.

In [12]:
# get timestamps

event_times = pd.read_csv("ctao_event_times_only.csv", parse_dates=["timestamp_1", "timestamp_2"])
two_times = event_times[event_times.file_name == filepath.name].iloc[0][["timestamp_1", "timestamp_2"]]
timestamp_1 = two_times["timestamp_1"].to_pydatetime()
timestamp_2 = two_times["timestamp_2"].to_pydatetime()

Define three time bins before the T0 of the events. These time bins will be set at zero flux:

In [13]:
delta = (np.log10(times[-1][1]) - np.log10(times[0][0])) / len(times)
n_time_extra = 3
new_times = np.logspace(np.log10(times[0][0]) - delta * n_time_extra, np.log10(times[-1][1]), 71 + n_time_extra)

Let's create the gammapy format of the model. Define a soure position and the gammapy axes for times and energies, and then fill only the epochs with flux different from zero:

In [14]:
pointing_position = SkyCoord(ra, dec, frame="icrs")

position = FixedPointingInfo(fixed_icrs=pointing_position.icrs)

# time axis
#time_axis = MapAxis.from_bounds(new_times[0], new_times[-1], nbin=73, name="time", interp="log")
time_axis = MapAxis.from_edges(new_times, unit="s", name="time", interp="log")

# energy axis
energy_axis = MapAxis.from_nodes(
    energies['Energies'], unit=energies['Energies'].unit, name="energy"
)

# create the spectrum in the new time range
spec_mod = np.lib.recfunctions.structured_to_unstructured(spectra.as_array())
newspec = np.zeros( (len(time_axis.center) , len(energy_axis.center)))

# fill the spectrum
newspec[n_time_extra:,:] = spec_mod

Make the Gammapy maps of the model. Please, be careful with the spectral units:

In [15]:
# create the RegionNDMap containing fluxes
m = RegionNDMap.create(
    region=PointSkyRegion(center=pointing_position),
    axes=[energy_axis, time_axis],
    unit="cm-2 s-1 GeV-1",
)

# to compute the spectra as a function of time we extract the coordinates of the geometry
coords = m.geom.get_coord(sparse=True)

# We reshape the spectrum array to perform broadcasting
newspec = newspec.reshape(m.data.shape)

# evaluate the spectra and fill the RegionNDMap
m.quantity = newspec * u.cm**-2 * u.s**-1 * u.GeV**-1

m

In [10]:
temporal_model_ref = Time(timestamp_1, format="datetime", scale="utc")
new_filename = Path("/Users/jarred/Work/ctao-science-data-challenge/gammapy_models") / (filepath.stem + "_gammapy.fits")
temp = LightCurveTemplateTemporalModel(m, t_ref=temporal_model_ref, filename=new_filename, method="linear", values_scale="log")
#temp.method = "log"
#temp.values_scale = "log"
temp.write(new_filename, format="map", overwrite=True)

confirm that it works

In [11]:
temporal_model = LightCurveTemplateTemporalModel.read(new_filename, format="map")

## Function that converts a FITS file to a gammapy-compatible format

In [69]:
# function that converts a FITS file to a gammapy-compatible format

def convert_to_gammapy(
    event_id: int,
    model_filepath: Path | str,
    timestamp: pd.Timestamp | np.datetime64,
    output_dir: Path | str = "./gammapy_models",
):
    
    model_filepath = Path(model_filepath).absolute()
    
    output_dir = Path(output_dir).absolute()
    
    if not output_dir.exists():
        output_dir.mkdir(parents=True, exist_ok=True)
    
    energies = Table.read(model_filepath, format="fits", hdu=1)
    times = Table.read(model_filepath, format="fits", hdu=2)
    spectra = Table.read(model_filepath, format="fits", hdu=3)

    # Get header information from primary HDU
    header = fits.getheader(model_filepath, ext=0)

    latitutde = header['LAT'] * u.rad
    longitude = header['LONG'] * u.rad
    distance = (header['DISTANCE'] * u.kpc).to(u.Mpc)
    redshift = Distance(distance).z

    ra = longitude.to(u.deg)
    dec = latitutde.to(u.deg)
    
    delta = (np.log10(times[-1][1]) - np.log10(times[0][0])) / len(times)
    n_time_extra = 3
    new_times = np.logspace(np.log10(times[0][0]) - delta * n_time_extra, np.log10(times[-1][1]), 71 + n_time_extra)
    
    
    pointing_position = SkyCoord(ra, dec, frame="icrs")

    # time axis
    #time_axis = MapAxis.from_bounds(new_times[0], new_times[-1], nbin=73, name="time", interp="log")
    time_axis = MapAxis.from_edges(new_times, unit="s", name="time", interp="log")

    # energy axis
    energy_axis = MapAxis.from_nodes(
        energies['Energies'], unit=energies['Energies'].unit, name="energy"
    )

    # create the spectrum in the new time range
    spec_mod = np.lib.recfunctions.structured_to_unstructured(spectra.as_array())
    newspec = np.zeros( (len(time_axis.center) , len(energy_axis.center)))

    # fill the spectrum
    newspec[n_time_extra:,:] = spec_mod
    
    # create the RegionNDMap containing fluxes
    m = RegionNDMap.create(
        region=PointSkyRegion(center=pointing_position),
        axes=[energy_axis, time_axis],
        unit="cm-2 s-1 GeV-1",
    )

    # We reshape the spectrum array to perform broadcasting
    newspec = newspec.reshape(m.data.shape)

    # evaluate the spectra and fill the RegionNDMap
    m.quantity = newspec * u.cm**-2 * u.s**-1 * u.GeV**-1
    
    temporal_model_ref = Time(timestamp, scale="utc")
    new_filename = output_dir / (str(event_id) + "_" + model_filepath.stem + "_gammapy.fits")
    temp = LightCurveTemplateTemporalModel(m, t_ref=temporal_model_ref, filename=new_filename, method="linear", values_scale="log")

    temp.write(new_filename, format="map", overwrite=True)
    
    # Add distance and redshift metadata to the FITS header
    with fits.open(new_filename, mode='update') as hdul:
        hdr = hdul[0].header
        hdr['DISTANCE'] = (distance.value, f'Distance in {distance.unit}')
        hdr['REDSHIFT'] = (redshift.value, 'Redshift')
        hdul.flush()
    
    return new_filename

In [70]:
catalog = Path('/Users/jarred/Downloads/TeV Catalog O5')
events_csv = "/Users/jarred/Work/ctao-science-data-challenge/notebooks/ctao_event_times.csv"

events = pd.read_csv(events_csv, parse_dates=["timestamp"])



In [71]:
output = events.apply(
    lambda row: convert_to_gammapy(
        event_id=row.superevent_id,
        model_filepath=catalog / row.file_name,
        timestamp=row.timestamp,
        output_dir=Path("/Users/jarred/Work/ctao-science-data-challenge/gammapy_models")
    ),
    axis=1
)